### Setting up RAG
Set up the OpenAI client:

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

Create the assistant:

In [4]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

#### Testing it
Let's try a question:

In [5]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally:

1. Install Ollama from: https://ollama.com/download  
   - macOS: download the `.pkg`
   - Windows: download the `.msi`
   - Linux: run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. Open a terminal and run:
```bash
ollama run llama3
```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

3. To test that the local server is running, use:
```bash
curl http://localhost:11434
```

If you want to use it from Python, install the client with:
```bash
pip install ollama
```


This works fine. The search finds relevant FAQ entries about Ollama, and the LLM gives a good answer.

Now try something slightly different:

In [6]:
assistant.rag("How do I run Olama locally?")

'I don’t see any information in the context about **Olama** specifically.\n\nIf you meant running the course **locally**, the FAQ says you can do that if you’re comfortable setting up:\n\n- Python\n- `uv`\n- Jupyter\n- Docker\n- any other tools needed for the module\n\nIt also says to **document your setup** and keep your environment **reproducible**.\n\nIf you meant something else by “Olama,” please clarify.'

The word "Olama" doesn't match "Ollama" in our index. We use lexical search, so it looks for the exact word and finds nothing. The LLM gets these bad results and either says "I don't know" or answers with irrelevant information.

This is the limitation of a fixed pipeline. The search runs once with the exact query the user typed, and there's no second chance. The pipeline doesn't know the search failed, so it can't try again with a corrected query.

We need something smarter. We need an agent.

#### The agent alternative
An agent puts the LLM in charge.

Instead of running search ourselves, we give the LLM a `search` tool. It decides when to call it and what to search for.

The same typo question now goes like this:

flowchart TD
    U([User: How do I run Olama?])
    L1[LLM: I'll search for 'Olama']
    S1[search - Olama - no useful results]
    L2[LLM: Hmm, no results. Maybe a typo for 'Ollama'?]
    S2[search - Ollama - found results!]
    A([LLM: Here's how to run Ollama locally...])

    U --> L1 --> S1 --> L2 --> S2 --> A


The LLM searched, saw the results were bad, and decided to try again with a different query. It made that decision on its own. We didn't write any code to handle typos.

The difference is about who makes the decisions:

- With RAG, the developer decides. We fix the steps up front, so search always runs once with the exact user query.
- With an agent, the LLM decides. It chooses which actions to take and when to stop.

The mechanism that makes this possible is function calling, and that's what the rest of this lesson is about.

#### Asking without tools
First, let's see what the LLM does without any tools. We ask it a course-specific question and look at the answer.

In [7]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Maybe — but I need a bit more context to tell you for sure.\n\nPlease tell me:\n1. **Which course** you mean\n2. **What platform or school** it’s on\n3. Whether you want to join **right now** or for a **future session**\n\nIn general, you can usually join a course if:\n- enrollment is still open,\n- you meet any prerequisites,\n- and there are remaining spots or the course allows self-enrollment.\n\nIf you want, send me the course name or a link and I’ll help you figure out whether you can still join.'

The model answers from its general knowledge, something like "it depends on the course" or "check the course website". It doesn't know about our FAQ, so the answer is vague and not helpful. This is exactly why we need RAG, and why we want to hand the model a tool.

#### Defining the tool
First we define a top-level `search` function that queries the `index` directly. The model will reference it by this name. We keep the Python function and the tool name aligned so the dispatch is easier later.

In [8]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

Next we tell the model about this function. The model doesn't see our Python code, only a schema describing what the function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java.

In [9]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

The `description` is the most important field, because the model reads it to decide when to call the function. `parameters` is a JSON schema for the arguments, and we mark `query` as required so the model always fills it in.

#### Sending the question with the tool
Now we send the same question as before, but this time we include the tool in the request:

In [10]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered course join after start enroll anytime"}', call_id='call_CfxYHhcmqZ6wArsxTr2g9LWT', name='search', type='function_call', id='fc_03539b540765ec94006a5fb66f3b08819db37f72c849ae1a95', caller=None, namespace=None, status='completed')]

Look at the output. Instead of a message with the answer, the response contains a `function_call` entry. The model decided it needs to search the FAQ before answering. Rather than reply, it asked us to run the search function first.

Look at the arguments too. The model didn't pass our question verbatim. It judged the raw question wasn't the best query to search with. So it rewrote our enrollment question into search keywords like "enroll late join course".

#### Executing the function and sending the result back
The function call contains JSON arguments. We parse them, call our search function, and serialize the result.

In [11]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [12]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How should I start the course and follow the weekly workflow?",
    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

Now we send this result back to the model. First, we add the model's output to the conversation history - the model needs to see its own function call. Then we add the tool result.

In [13]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

The `call_id` links the tool result to the specific function call the model requested. If the model makes multiple function calls in one turn, each one gets its own `call_id`.

#### Asking the model again
We call the API a second time with the expanded history:

In [14]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join.\n\nYou can start anytime, and the course materials are available. If you want a certificate, though, you’ll need to submit your project while submissions are still open.'

This time the model has the original question, its own decision to call `search`, and the FAQ results. It can now produce a proper course-specific answer.

We have to send the whole history because LLMs are stateless between API calls. The memory is the list you send as `input`. If you send only the tool result, the model has no idea what's going on. So on this second call we replay everything we have so far. That means the question, the decision to call `search`, and the result we got back.

That's the full function-calling loop for a single turn. With plain RAG we made one call, and here we make two. Turning RAG agentic means more round-trips.

People call this pattern "agentic RAG", "tool use", or "function calling". The idea behind all of them is the same. The LLM decides which tools to call.

#### Token usage and cost
We just made two API calls instead of one. Each call we send to the model costs money, so it's worth checking how much one tool-using turn actually costs.

The response has a `usage` field with the token counts:

In [15]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(775, 44)

For each model the provider publishes a price per million input tokens and per million output tokens. Plug those numbers in to convert tokens to dollars.

In [17]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(775, 44)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.00014265


This usage is only for the second API call. The first call also has its own usage and its own cost. That was the call where the model decided to invoke `search`. Two calls means we pay twice. We pay even more on the second call, because we resend the full history as input.

### The Agentic Loop

Previously, we did function calling by hand. We sent a message and got back a function call. We ran it, sent the result back, and got the answer.

That works for one function call. It breaks down when the model wants to search several times, or when the first search misses the answer. We don't know in advance how many calls the model will want. So we need a loop that keeps calling the model and running tools until it's done. An agent is exactly that.

#### Anatomy of an agent
With the LLM in the driver's seat, we have an agent. It's an AI assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the `developer` message. The better the instructions, the better the agent helps.
- Tools, the functions the agent can call to carry out the task. For us that's only `search`.
- Memory, the message history. We append every prompt, every model output, and every tool result. The agent reads this to know what it has already tried.

Everything below is the code that wires these three together inside a loop.

#### A developer prompt
So far we've relied on the model to figure out when to search. We make that more reliable with a `developer` message that spells out how to behave. This is where we give the agent its role. The same message also pushes it toward multiple searches, so we get to watch the loop run more than once.

In [18]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

#### A function-call helper
We'll be running function calls repeatedly inside the loop, so let's wrap that in a small helper. It turns the JSON arguments into a Python dict, calls the right function, and serializes the result. We only have one tool for now, so we dispatch on the function name directly.

In [19]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        results = search(**args)
    
    result_json = json.dumps(results, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

The helper returns the exact structure the Responses API expects. When we add more tools later, we'll extend this with more `if` branches (or switch to a registry).

#### Processing one response
Let's process a single model response. We append each output entry to the conversation, print any messages, and run any function calls. Function-call results get appended too.

In [20]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered late enroll register course FAQ"}


The `has_function_calls` flag tells us whether the model needs another API call. If the response contains a function call, the updated `messages` has tool output the model hasn't seen yet. We'll need to send it back.

#### The full agent loop
We wrap this in a `while` loop. The loop keeps calling the model until it returns a response without any function calls. We also keep an iteration counter so we can see how many round-trips happened.

In [ ]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

This is the core agent loop. The model reasons about the next action. Your code performs it, and the model sees the result on the next turn. The loop stops when the model returns a final answer with no more tool calls.

We don't decide how many times the model searches. The model does, and we keep looping until it stops asking for tools.

The exit condition is the simplest one possible. No function calls this turn means we're done. Other frameworks add safety nets on top, like a max iteration count, a token budget, or a wall-clock limit. You might cap it at five iterations and force an answer on the last one. The core is still this one flag.

#### Wrapping it in a function
Let's wrap the loop in a function so we can reuse it. The function takes the instructions and the question as parameters, and returns the final answer.

In [ ]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

Try it with a question that has a typo:

In [ ]:
agent_loop(instructions, "How do I run Olama locally?")

Watch what happens. The agent searches for "Olama" and gets poor results. It then searches again with "Ollama" and finds the answer. The loop lets the model recover from a bad search on its own. That's the whole point of going agentic.

Also try the course enrollment question:

In [ ]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

#### Encouraging multiple searches
There's a subtle issue here. The model often answers after the first search, even when more searches would help. It reasons that it already knows enough, so why bother. We push it to explore more by rewriting the instructions.

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

Now the agent makes multiple searches per question and doesn't stop after the first round of results. The instructions are how we steer the agent. It can still decide to skip ahead sometimes, so don't expect it to follow them every single run.

#### Restricting off-topic questions
Right now the agent will answer anything. Ask it about chess and it will still try.

In [ ]:
agent_loop(instructions, "what's queen gambit?")

We want a course assistant, not a general chatbot. We tighten the instructions so the agent only answers from the FAQ. For our own use we might be fine letting it answer from general knowledge. So treat this mainly as an illustration of steering scope.

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

This is a lightweight form of an input guardrail. We tell the agent what's in scope and what isn't. A real guardrail checks the input before the agent runs and can block off-topic questions outright. That's a separate topic, but instructions are the first place to start.

This handwritten loop is the best way to understand what frameworks hide from you. Every agent framework wraps this same pattern, whether it's LangChain, PydanticAI, or the OpenAI Agents SDK.

#### ToyAIKit
The handwritten agent loop from the previous lesson is educational but repetitive. Every time you build a new agent, you'd write the same while-loop, the same function-call handling, the same message management.

ToyAIKit wraps this pattern so you can focus on tools, prompts, and behavior. We built it together in a DataTalks.Club workshop a while back. It does the same thing as our handwritten loop with less boilerplate. If you open its runners code, you'll find the same while True loop we wrote by hand.

ToyAIKit is a teaching and experimentation library, and it is NOT meant for production use. We use it because it's minimal and you can see what it does.

Import the classes we need:

In [21]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

#### Registering the tool
We register our `search` function along with the schema from earlier lessons:

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

#### Letting ToyAIKit generate the schema
Writing that schema by hand is annoying, and we don't want to do it for every function. So we don't have to.

If we add a type hint and a docstring to search, ToyAIKit reads them and derives the schema for us:

In [ ]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

Then register it without passing a schema:

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search)

You can look at what ToyAIKit produced.

In [ ]:
agent_tools.get_tools()

The output is the same JSON schema we hand-wrote in the function calling lesson. ToyAIKit generated it from the docstring and the type hint.

Every modern agent framework does this same trick. It reads a typed Python function with a docstring and builds the schema from it. The OpenAI Agents SDK, PydanticAI, LangChain and Google ADK all work this way. You write the tool and the framework figures out how to describe it.

#### The chat interface and runner
Create the chat interface and a callback, then build the runner:

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

The chat_interface handles display in the notebook. The callback renders model messages and tool calls as they happen. The runner runs the agent loop, the same while True we wrote by hand. It sends messages, executes function calls, adds tool outputs back, and repeats until the model is done.

We pick gpt-5.4-mini here on purpose. Without it, ToyAIKit falls back to a smaller, faster default that doesn't follow the instructions as reliably.

#### Running one prompt
Run a single prompt:

In [ ]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

We used the typo "Olama" on purpose. The agent searches and gets poor results, then retries with "Ollama". The recovery is the same as the handwritten loop. The notebook output is nicer to watch. Each tool call and message renders inline, so you can look at every search result.

The `result` is a `LoopResult` with `all_messages` (the full conversation), token counts, and `cost` (computed from token usage).

#### Cost and tokens
Look at what the call cost:

In [ ]:
result.cost

Useful while developing - especially with multi-turn agents where one prompt can trigger several model calls. The handwritten loop made you compute this by hand. The framework keeps a running total for you.

You can also look at the full message history.

In [ ]:
result.all_messages

This is just a list - the same messages list we maintained by hand.

#### Continuing the conversation
Take the messages from the previous result and pass them as previous_messages on the next loop call:

In [ ]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

The runner picks up where the last call left off, with the same agent loop and an extended history. The model knows "different model" refers to Ollama because it sees the previous turn in memory. Without that history, it would have no idea what we're asking about.

#### Interactive chat
For a chat-like workflow, run the built-in input loop:

In [ ]:
runner.run()

Type questions and get answers. Type "stop" to exit.